### Since governor elections are recorded at the county-level, we first wanted to see the performace of each respective county, and then check how the voting patterns add up to the overall state election outcome.

#### Specifically, we are looking at whether the political party that wins the most influential counties also happens to win the state as a whole. Furthermore, we define "influential" counties by determing the counties with the largest share of total votes in each state.

In [2]:
# importing packages
import numpy as np
import pandas as pd

# reading in files
governors_county_candidate = pd.read_csv("4-US_Election_2020/governors_county_candidate.csv")
governors_county = pd.read_csv("4-US_Election_2020/governors_county.csv")
governors_state = pd.read_csv("4-US_Election_2020/governors_state.csv")

president_county_candidate = pd.read_csv("4-US_Election_2020/president_county_candidate.csv")
president_county = pd.read_csv("4-US_Election_2020/president_county.csv")
president_state = pd.read_csv("4-US_Election_2020/president_state.csv")

#### **Sub Question 1:** Which counties are the most influential in each state's gubernatorial election based on their share of total votes?

In [3]:
county_votes = (governors_county
                .groupby(["state", "county"])
                .agg(total_votes=("total_votes", "max"))
                .reset_index())

state_votes = governors_state.copy()
df = county_votes.merge(state_votes, on="state")
df["vote_share"] = df["total_votes"] / df["votes"]

df_top = (df
          .sort_values(["state", "vote_share"], ascending=[True, False])
          .groupby("state")
          .head(5))

df_top_sorted = df_top.sort_values(
    by=["state", "vote_share"], 
    ascending=[True, False]
).reset_index(drop=True)

df_top_sorted[["state", "county", "vote_share"]]

,state,county,vote_share
0,Delaware,New Castle County,0.583866
1,Delaware,Sussex County,0.262572
2,Delaware,Kent County,0.176652
3,Indiana,Marion County,0.129405
4,Indiana,Lake County,0.072824
5,Indiana,Hamilton County,0.064230
6,Indiana,Allen County,0.055980
7,Indiana,St. Joseph County,0.038153
8,Missouri,St. Louis County,0.178087
9,Missouri,Jackson County,0.110898


##### We calculated each county's share of total votes within it's state to determine its level of influence in the gubernatorial election. Each county was ranked by it's total vote share, and then the top 5 counties in each state were chosen as being the most influential as they provide the larget amount of votes to the total election results for their state.

#### **Sub Question 2:** Which political party tends to dominate in the most influential counties across each state?

In [4]:
state = governors_state

data = county_votes.merge(state, on="state")

data["share"] = data["total_votes"] / data["votes"]

top = (data
       .sort_values(["state", "share"], ascending=[True, False])
       .groupby("state")
       .head(5))

winners = governors_county_candidate[
    governors_county_candidate["won"] == True
]

top_party = top.merge(winners, on=["state", "county"])

top_party = top_party.rename(columns={"party_y": "party"})

top_party_sorted = (top_party
                    .sort_values(["state", "share"], ascending=[True, False])
                    .reset_index(drop=True))

top_party_sorted[["state", "county", "party"]]

,state,county,party
0,Delaware,New Castle County,DEM
1,Delaware,Sussex County,REP
2,Delaware,Kent County,DEM
3,Indiana,Marion County,DEM
4,Indiana,Lake County,DEM
5,Indiana,Hamilton County,REP
6,Indiana,Allen County,REP
7,Indiana,St. Joseph County,REP
8,Missouri,St. Louis County,DEM
9,Missouri,Jackson County,DEM


#### After determing the top 5 most influential counties in each state's gubernatorial elections based on vote share, we then determined the political party of those counties. This allows us to see which party dominates in the most influential counties across each state.

#### **Sub Question 3:** Do the political party preferences of the top 5 counties match the overall state election outcome?

In [5]:
data = county_votes.merge(governors_state, on="state")
data["share"] = data["total_votes"] / data["votes"]

top = (data
       .sort_values(["state", "share"], ascending=[True, False])
       .groupby("state")
       .head(5))

winners = governors_county_candidate[
    governors_county_candidate["won"] == True
]

top_party = top.merge(winners, on=["state", "county"])

top5_party = (top_party
              .groupby(["state", "party"])
              .size()
              .reset_index(name="count")
              .sort_values(["state", "count"], ascending=[True, False])
              .groupby("state")
              .head(1)
              .rename(columns={"party": "top5_party"}))

state_winner = (governors_county_candidate
                .groupby(["state", "party"])["votes"]
                .sum()
                .reset_index()
                .sort_values(["state", "votes"], ascending=[True, False])
                .groupby("state")
                .head(1)
                .rename(columns={"party": "state_winner"}))

result = top5_party.merge(state_winner[["state", "state_winner"]], on="state")
result["aligns"] = result["top5_party"] == result["state_winner"]

display(result)

,state,top5_party,count,state_winner,aligns
0,Delaware,DEM,2,DEM,True
1,Indiana,REP,3,REP,True
2,Missouri,DEM,3,REP,False
3,Montana,DEM,3,REP,False
4,New Hampshire,REP,5,REP,True
5,North Carolina,DEM,5,DEM,True
6,North Dakota,REP,5,REP,True
7,Utah,REP,5,REP,True
8,Vermont,REP,4,REP,True
9,Washington,DEM,4,DEM,True


#### We then compared the dominant political party in the top 5 most influential counties to the overall winner in the gubertorial elections for each state. We did this as it allowed us to determine whether the voting patterns in the deemed most impactful counties align witht the statewide results. Out of the 11 states listed, 9 states showed alignment, illustrating that the political party dominating the most influential counties is mostly reflected in the overall results.

#### **Overall:** Do the most influential counties in gubernatorial elections also reflect the outcome of presidential elections in each state? 

In [6]:
pres_winner = (president_county_candidate
               .groupby(["state", "party"])["total_votes"]
               .sum()
               .reset_index()
               .sort_values(["state", "total_votes"], ascending=[True, False])
               .groupby("state")
               .head(1)
               .rename(columns={"party": "pres_winner"}))

result = result.merge(pres_winner[["state", "pres_winner"]], on="state")
result["aligns_president"] = result["top5_party"] == result["pres_winner"]
result[["state", "top5_party", "state_winner", "pres_winner", "aligns", "aligns_president"]]

,state,top5_party,state_winner,pres_winner,aligns,aligns_president
0,Delaware,DEM,DEM,DEM,True,True
1,Indiana,REP,REP,REP,True,True
2,Missouri,DEM,REP,REP,False,False
3,Montana,DEM,REP,REP,False,False
4,New Hampshire,REP,REP,DEM,True,False
5,North Carolina,DEM,DEM,REP,True,False
6,North Dakota,REP,REP,REP,True,True
7,Utah,REP,REP,REP,True,True
8,Vermont,REP,REP,DEM,True,False
9,Washington,DEM,DEM,DEM,True,True


#### We further compared whether this earlier alignment reflected amongst the presidential elections. While the top 5 counties did sway the gubertorial elections to their dominant political party preference, this wasn't true for the presidential elections. Matter of fact, there's no strong correlation between the most influential counties' political party preferences and the outcome of the presidental election. This could be explained through the differences in impacts counties have on a their local vs. national elections. Furthermore, this highlights the importance of the public vote on a local and state level compared to the electoral college system the presidential election employs. 